# M3 Hadoop MapReduce — VS Code / Jupyter Practice Notebook

This notebook is designed to run **line by line** in VS Code or Jupyter.

It uses the same mapper/reducer logic:

- `mapper2.py`
- `reducer.py`

It does **not** start Hadoop Streaming, so it will not hang forever. It runs mapper and reducer locally using Python, then saves the result to the output folder.


## Step 1 — Import libraries


In [1]:
from pathlib import Path
import subprocess
import sys
import shutil


## Step 2 — Set project paths


In [2]:
PROJECT_ROOT = Path("/Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow")
ASSIGNMENTS_DIR = PROJECT_ROOT / "Assignments"
CODEBOOK_DIR = ASSIGNMENTS_DIR / "Codebook"
DATASET_DIR = ASSIGNMENTS_DIR / "Dataset"
OUTPUTS_DIR = ASSIGNMENTS_DIR / "Outputs"

MAPPER = CODEBOOK_DIR / "mapper2.py"
REDUCER = CODEBOOK_DIR / "reducer.py"
INPUT_FILE = DATASET_DIR / "input.txt"
OUTPUT_FILE = OUTPUTS_DIR / "m3_wordcount_output_local.txt"

print("Codebook folder:", CODEBOOK_DIR)
print("Dataset folder:", DATASET_DIR)
print("Outputs folder:", OUTPUTS_DIR)


Codebook folder: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook
Dataset folder: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Dataset
Outputs folder: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Outputs


## Step 3 — Create folders if needed


In [3]:
CODEBOOK_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Folders are ready.")


Folders are ready.


## Step 4 — Create sample input file if it does not exist


In [5]:
if not INPUT_FILE.exists():
    INPUT_FILE.write_text(
        """Hadoop is useful for big data.

Hadoop MapReduce uses mapper and reducer files.

This assignment runs mapper2.py and reducer.py.
""",
        encoding="utf-8",
    )

    print("Created sample input file:", INPUT_FILE)

else:
    print("Using existing input file:", INPUT_FILE)


Using existing input file: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Dataset/input.txt


## Step 5 — Check mapper and reducer files


In [7]:
missing = []

if not MAPPER.exists():
    missing.append(str(MAPPER))

if not REDUCER.exists():
    missing.append(str(REDUCER))

if missing:
    raise FileNotFoundError(
        "Missing required file(s):\n"
        + "\n".join(missing)
        + "\n\nPlease place mapper2.py and reducer.py in:\n"
        + str(CODEBOOK_DIR)
    )

print("Mapper found:", MAPPER)
print("Reducer found:", REDUCER)


Mapper found: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook/mapper2.py
Reducer found: /Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Codebook/reducer.py


## Step 6 — Clean previous output


In [8]:
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()
    print("Previous output removed.")
else:
    print("No previous output file found.")


Previous output removed.


## Step 7 — Define helper function to run a command


In [9]:
def run_command(command, input_text: str, timeout_seconds: int = 30):
    return subprocess.run(
        command,
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
        timeout=timeout_seconds,
    )


## Step 8 — Run mapper2.py


In [11]:
input_text = INPUT_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

mapper_result = run_command(
    [sys.executable, str(MAPPER)],
    input_text=input_text
)

if mapper_result.returncode != 0:
    raise RuntimeError(
        "Mapper failed:\n"
        + mapper_result.stderr
    )

print("Mapper ran successfully.")
print("\nFirst mapper output lines:\n")

print(
    "\n".join(
        mapper_result.stdout.splitlines()[:20]
    )
)


Mapper ran successfully.

First mapper output lines:

hadoop	1
useful	1
big	1
data	1
hadoop	1
mapreduce	1
uses	1
mapper	1
reducer	1
files	1
assignment	1
runs	1
mapper2	1
py	1
reducer	1
py	1


## Step 9 — Sort mapper output

Hadoop normally sorts mapper output before sending it to the reducer. Since this notebook runs locally, we sort it here.


In [13]:
sorted_mapper_output = "\n".join(
    sorted(mapper_result.stdout.splitlines())
)

print("Sorted mapper output sample:\n")

print(
    "\n".join(
        sorted_mapper_output.splitlines()[:20]
    )
)


Sorted mapper output sample:

assignment	1
big	1
data	1
files	1
hadoop	1
hadoop	1
mapper	1
mapper2	1
mapreduce	1
py	1
py	1
reducer	1
reducer	1
runs	1
useful	1
uses	1


## Step 10 — Run reducer.py


In [15]:
reducer_result = run_command(
    [sys.executable, str(REDUCER)],
    input_text=sorted_mapper_output
)

if reducer_result.returncode != 0:
    raise RuntimeError(
        "Reducer failed:\n"
        + reducer_result.stderr
    )

print("Reducer ran successfully.")
print("\nFirst reducer output lines:\n")

print(
    "\n".join(
        reducer_result.stdout.splitlines()[:20]
    )
)


Reducer ran successfully.

First reducer output lines:

assignment	1
big	1
data	1
files	1
hadoop	2
mapper	1
mapper2	1
mapreduce	1
py	2
reducer	2
runs	1
useful	1
uses	1


## Step 11 — Save output file


In [16]:
OUTPUT_FILE.write_text(reducer_result.stdout, encoding="utf-8")

print("Output saved to:")
print(OUTPUT_FILE)


Output saved to:
/Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Outputs/m3_wordcount_output_local.txt


## Step 12 — Display final WordCount output


In [17]:
print("================ WORD COUNT OUTPUT ================")
print(OUTPUT_FILE.read_text(encoding="utf-8", errors="ignore"))


================ WORD COUNT OUTPUT ================
assignment	1
big	1
data	1
files	1
hadoop	2
mapper	1
mapper2	1
mapreduce	1
py	2
reducer	2
runs	1
useful	1
uses	1



# Student Practice Section

The following cells are added **after the full working code**.

Students can practice by editing the input text, running mapper, sorting mapper output, running reducer, and saving/displaying the output.

Instructor note: the answer code is included below. You can comment it out or hide it the same way you did for Day 2.


## Practice 1 — Student updates the input text

Students should replace the text below with their own 5 or more lines.


In [ ]:
# ================= STUDENT PRACTICE 1 =================
# TODO: Replace the text below with your own paragraph.

practice_text = """
Hadoop processes large data files.
MapReduce separates work into map and reduce steps.
The mapper creates word and count pairs.
The reducer adds counts for each word.
Python can simulate MapReduce logic locally.
"""

# PRACTICE_INPUT_FILE = (
#     DATASET_DIR / "student_practice_input.txt"
# )

# PRACTICE_INPUT_FILE.write_text(
#     practice_text.strip() + "\n",
#     encoding="utf-8"
# )

# print("Practice input file created:")
# print(PRACTICE_INPUT_FILE)

# print("\nPractice input text:\n")

# print(
#     PRACTICE_INPUT_FILE.read_text(
#         encoding="utf-8"
#     )
# )


Practice input file created:
/Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Dataset/student_practice_input.txt

Practice input text:

Hadoop processes large data files.
MapReduce separates work into map and reduce steps.
The mapper creates word and count pairs.
The reducer adds counts for each word.
Python can simulate MapReduce logic locally.



## Practice 2 — Student runs mapper2.py on practice input


In [ ]:
# ================= STUDENT PRACTICE 2 =================
# TODO: Read the practice input file and pass it into mapper2.py.

# practice_input_text = PRACTICE_INPUT_FILE.read_text(
#     encoding="utf-8",
#     errors="ignore"
# )

# practice_mapper_result = run_command(
#     [sys.executable, str(MAPPER)],
#     input_text=practice_input_text
# )

# if practice_mapper_result.returncode != 0:
#     raise RuntimeError(
#         "Practice mapper failed:\n"
#         + practice_mapper_result.stderr
#     )

print("Practice mapper output sample:\n")

print(
    "\n".join(
        practice_mapper_result.stdout.splitlines()[:30]
    )
)


Practice mapper output sample:

hadoop	1
processes	1
large	1
data	1
files	1
mapreduce	1
separates	1
work	1
map	1
reduce	1
steps	1
mapper	1
creates	1
word	1
count	1
pairs	1
reducer	1
adds	1
counts	1
word	1
python	1
simulate	1
mapreduce	1
logic	1
locally	1


## Practice 3 — Student sorts mapper output


In [ ]:
# ================= STUDENT PRACTICE 3 =================
# TODO: Sort mapper output before sending it to reducer.

# practice_sorted_mapper_output = "\n".join(
#     sorted(
#         practice_mapper_result.stdout.splitlines()
#     )
# ) + "\n"

# print("Practice sorted mapper output sample:\n")

# print(
#     "\n".join(
#         practice_sorted_mapper_output.splitlines()[:30]
#     )
# )

Practice sorted mapper output sample:

adds	1
count	1
counts	1
creates	1
data	1
files	1
hadoop	1
large	1
locally	1
logic	1
map	1
mapper	1
mapreduce	1
mapreduce	1
pairs	1
processes	1
python	1
reduce	1
reducer	1
separates	1
simulate	1
steps	1
word	1
word	1
work	1


## Practice 4 — Student runs reducer.py


In [ ]:
# ================= STUDENT PRACTICE 4 =================
# TODO: Send sorted mapper output into reducer.py.

# practice_reducer_result = run_command(
#     [sys.executable, str(REDUCER)],
#     input_text=practice_sorted_mapper_output
# )

# if practice_reducer_result.returncode != 0:
#     raise RuntimeError(
#         "Practice reducer failed:\n"
#         + practice_reducer_result.stderr
#     )

# print("Practice reducer output:\n")

# print(
#     practice_reducer_result.stdout
# )


Practice reducer output:

adds	1
count	1
counts	1
creates	1
data	1
files	1
hadoop	1
large	1
locally	1
logic	1
map	1
mapper	1
mapreduce	2
pairs	1
processes	1
python	1
reduce	1
reducer	1
separates	1
simulate	1
steps	1
word	2
work	1



## Practice 5 — Student saves practice output


In [ ]:
# ================= STUDENT PRACTICE 5 =================
# TODO: Save reducer output to a practice output file.

# PRACTICE_OUTPUT_FILE = OUTPUTS_DIR / "m3_student_practice_output.txt"
# PRACTICE_OUTPUT_FILE.write_text(practice_reducer_result.stdout, encoding="utf-8")

# print("Practice output saved to:")
# print(PRACTICE_OUTPUT_FILE)


Practice output saved to:
/Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments/Outputs/m3_student_practice_output.txt


## Practice 6 — Student displays practice output


In [ ]:
# ================= STUDENT PRACTICE 6 =================
# TODO: Display the saved practice output file.

# print("================ STUDENT PRACTICE OUTPUT ================")
# print(PRACTICE_OUTPUT_FILE.read_text(encoding="utf-8", errors="ignore"))


================ STUDENT PRACTICE OUTPUT ================
adds	1
count	1
counts	1
creates	1
data	1
files	1
hadoop	1
large	1
locally	1
logic	1
map	1
mapper	1
mapreduce	2
pairs	1
processes	1
python	1
reduce	1
reducer	1
separates	1
simulate	1
steps	1
word	2
work	1



# Optional Challenge Practice

Students can try to modify the input text and answer:

1. Which word appears most often?
2. How many unique words are in the output?
3. What happens if punctuation or uppercase letters are added?


In [ ]:
# ================= OPTIONAL CHALLENGE =================
# Students may uncomment and run this after Practice 6.

# lines = PRACTICE_OUTPUT_FILE.read_text(encoding="utf-8").strip().splitlines()
# word_counts = []

# for line in lines:
#     word, count = line.split("	")
#     word_counts.append((word, int(count)))

# most_common_word = max(word_counts, key=lambda item: item[1])
# unique_word_count = len(word_counts)

# print("Most common word:", most_common_word)
# print("Number of unique words:", unique_word_count)
